<a href="https://colab.research.google.com/github/dheerajnalla09/AIML/blob/main/emotion_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Emotion Detection

In [3]:

import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2, ResNet50, EfficientNetV2B0

tf.keras.backend.clear_session()


In [4]:
from google.colab import files
uploaded = files.upload()

Saving emojis.zip to emojis.zip


In [5]:
!unzip "file name emojis.zip"

unzip:  cannot find or open file name emojis.zip, file name emojis.zip.zip or file name emojis.zip.ZIP.


In [6]:
!ls

emojis.zip  sample_data


In [7]:
!unzip emojis.zip

Streaming output truncated to the last 5000 lines.
  inflating: emojis/sad/im370.png    
  inflating: emojis/sad/im3700.png   
  inflating: emojis/sad/im3701.png   
  inflating: emojis/sad/im3702.png   
  inflating: emojis/sad/im3703.png   
  inflating: emojis/sad/im3704.png   
  inflating: emojis/sad/im3705.png   
  inflating: emojis/sad/im3706.png   
  inflating: emojis/sad/im3707.png   
  inflating: emojis/sad/im3708.png   
  inflating: emojis/sad/im3709.png   
  inflating: emojis/sad/im371.png    
  inflating: emojis/sad/im3710.png   
  inflating: emojis/sad/im3711.png   
  inflating: emojis/sad/im3712.png   
  inflating: emojis/sad/im3713.png   
  inflating: emojis/sad/im3714.png   
  inflating: emojis/sad/im3715.png   
  inflating: emojis/sad/im3716.png   
  inflating: emojis/sad/im3717.png   
  inflating: emojis/sad/im3718.png   
  inflating: emojis/sad/im3719.png   
  inflating: emojis/sad/im372.png    
  inflating: emojis/sad/im3720.png   
  inflating: emojis/sad/im3721.png   

In [8]:
img_size = (224,224)
batch_size = 32

In [10]:
data_dir = "/content/emojis"

In [11]:

# 🔥 Use tf.data API (always outputs RGB)
train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=img_size,
    batch_size=batch_size
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=img_size,
    batch_size=batch_size
)

class_names = train_ds.class_names
print("Classes:", class_names)


Found 28709 files belonging to 7 classes.
Using 22968 files for training.
Found 28709 files belonging to 7 classes.
Using 5741 files for validation.
Classes: ['angry', 'disgusted', 'fearful', 'happy', 'neutral', 'sad', 'surprised']


In [12]:
# Normalize
normalization_layer = layers.Rescaling(1./255)

train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))
val_ds = val_ds.map(lambda x, y: (normalization_layer(x), y))

for x, y in train_ds.take(1):
    print("Batch shape:", x.shape)


Batch shape: (32, 224, 224, 3)


In [13]:
def build_model(base_model):
    base_model.trainable = False

    model = models.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dense(128, activation='relu'),
        layers.Dense(len(class_names), activation='softmax')
    ])
    return model


In [14]:

models_dict = {
    "MobileNetV2": build_model(MobileNetV2(weights='imagenet', include_top=False, input_shape=(224,224,3))),
    "ResNet50": build_model(ResNet50(weights='imagenet', include_top=False, input_shape=(224,224,3))),
    "EfficientNetV2B0": build_model(EfficientNetV2B0(weights='imagenet', include_top=False, input_shape=(224,224,3)))
}

results = {}

for name, model in models_dict.items():
    print("Training:", name)
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    h = model.fit(train_ds, validation_data=val_ds, epochs=2)
    results[name] = h.history['val_accuracy'][-1]

print("Results:", results)


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
24274472/24274472 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Training: MobileNetV2
Epoch 1/2
718/718 ━━━━━━━━━━━━━━━━━━━━ 1057s 1s/step - accuracy: 0.4319 - loss: 1.4790 - val_accuracy: 0.4536 - val_loss: 1.4455
Epoch 2/2
718/718 ━━━━━━━━━━━━━━━━━━━━ 1108s 2s/step - accuracy: 0.4935 - loss: 1.3319 - val_accuracy: 0.4879 - val_loss: 1.3565
Training: ResNet50
Epoch 1/2
718/718 ━━━━━━━━━━━━━━━━━━━━ 4460s 6s/step - accuracy: 0.2345 - loss: 1.8295 - val_accuracy: 0.2472 - val_loss: 1.8284
Epoch 2/2
718/718 ━━━━━━━━━━━━━━━━━━━━ 4445s 6s/step - accuracy: 0.2541 - loss: 1.8015 - val_accuracy: 0.2491 - val_loss: 1.8057
Training: EfficientNetV2B0
Epoch 1/2
718/718 ━━━━━━━━━━━━━━━━━━━━ 1391s 2s/step - accuracy: 0.2462 - loss: 1.8200 - val_accuracy: 0.2453 - val_loss: 1.8150
Epoch 2/2
718/718 ━━━━━━━━━━━━━━━━━━━━ 1367s 2s/step - accuracy: 0.2518 - loss: 1.8134 - val_accuracy: 0.2453 - val_loss: 1.8164
Results

In [15]:

# Prediction function
def predict_image(img):
    img = tf.image.resize(img, img_size)
    img = tf.expand_dims(img/255.0, axis=0)
    preds = model.predict(img)
    return class_names[np.argmax(preds)]


In [16]:
# Gradio UI
import gradio as gr

best_model = list(models_dict.values())[0]

def gradio_predict(img):
    img = tf.image.resize(img, img_size)
    img = tf.expand_dims(img/255.0, axis=0)
    preds = best_model.predict(img)
    return class_names[np.argmax(preds)]

demo = gr.Interface(fn=gradio_predict, inputs=gr.Image(), outputs="label")
demo.launch()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f171283124face07b2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
